<a href="https://colab.research.google.com/github/Tifym7/AquaGraph/blob/moni/Cassini2026_Sentinel2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install earthengine-api geemap rasterio geopandas folium matplotlib numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 6.8 MB/s eta 0:00:00


In [ ]:
import ee
import geemap

#Initialize Earth Engine
try:
  ee.Initialize(project="cassini2026")
except Exception as e:
  ee.Authenticate()
  ee.Initialize(project="cassini2026")

print("Earth Engine initialized.")

Earth Engine initialized.


In [ ]:
ROMANIA = (
    ee.FeatureCollection("FAO/GAUL/2015/level0")
    .filter(ee.Filter.eq("ADM0_NAME", "Romania"))
    .geometry()
)


In [ ]:
def mask_s2_clouds(img):
    scl = img.select('SCL')
    mask = scl.neq(3).And(scl.neq(9)).And(scl.neq(8)).And(scl.neq(10))
    return img.updateMask(mask)

image = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(ROMANIA)
    .filterDate("2023-06-01", "2023-08-31")
    .map(mask_s2_clouds)
    .median()
    .clip(ROMANIA)
)

In [ ]:
def compute_indices(img):
    ndwi = img.normalizedDifference(['B3','B8']).rename('NDWI')
    mndwi = img.normalizedDifference(['B3','B11']).rename('MNDWI')
    ndvi = img.normalizedDifference(['B8','B4']).rename('NDVI')
    ndti = img.normalizedDifference(['B4','B3']).rename('NDTI')
    ndci = img.normalizedDifference(['B5','B4']).rename('NDCI')

    bsi = img.expression(
        '((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))',
        {
            'SWIR': img.select('B11'),
            'RED': img.select('B4'),
            'NIR': img.select('B8'),
            'BLUE': img.select('B2')
        }
    ).rename('BSI')

    turbidity = img.select('B4').rename('TURBIDITY')

    return ee.Image.cat([ndwi, mndwi, ndvi, ndti, ndci, bsi, turbidity])

indices = compute_indices(image).select([
    'NDWI','MNDWI','NDVI','NDTI','NDCI','BSI','TURBIDITY'
])

In [ ]:
rivers = ee.FeatureCollection("projects/cassini2026/assets/eu-hydro")

In [ ]:
rivers_stats = indices.reduceRegions(
    collection=rivers,
    reducer=ee.Reducer.mean(),
    scale=20,
)
rivers_stats = indices.reduceRegions(
    collection=rivers,
    reducer=ee.Reducer.mean(),
    scale=20,
)

def safe_number(feature, name):
    val = feature.get(name)
    return ee.Number(
        ee.Algorithms.If(
            ee.Algorithms.IsEqual(val, None),
            0,
            val
        )
    )

In [ ]:
def compute_risk(feature):
    mndwi = safe_number(feature, 'MNDWI')
    ndti = safe_number(feature, 'NDTI')
    turbidity = safe_number(feature, 'TURBIDITY')
    ndvi = safe_number(feature, 'NDVI')
    ndci = safe_number(feature, 'NDCI')
    bsi = safe_number(feature, 'BSI')

    is_water = mndwi.gt(0.2)

    water_risk = (
        ndti.gt(0.1).multiply(2)
        .add(turbidity.gt(0.15))
        .add(ndci.gt(0.1))
        .add(ndti.gt(0.1).And(ndci.gt(0.1)))
    ).multiply(is_water)

    land_risk = (
        ndvi.lt(0.3)
        .add(bsi.gt(0.3))
    )

    risk_score = water_risk.add(land_risk)

    return feature.set({
        'risk_score': risk_score,
        'water_risk': water_risk,
        'land_risk': land_risk,
        'is_water': is_water
    })

In [ ]:
def classify_risk(feature):
    score = ee.Number(feature.get('risk_score'))

    risk = ee.Algorithms.If(score.gte(5), 'HIGH',
           ee.Algorithms.If(score.gte(3), 'MEDIUM', 'LOW'))

    return feature.set({'risk_level': risk})

In [ ]:
rivers_risk = rivers_stats.map(compute_risk)
rivers_final = rivers_risk.map(classify_risk)

vis_params = {
    "min": 0,
    "max": 1,
    "palette": ["green", "yellow", "red"]
}

risk_image = stats.reduceToImage(
    properties=["risk"],
    reducer=ee.Reducer.first()
)

Map.centerObject(rivers, 7)
Map.addLayer(risk_image, vis_params, "Risk Map")
Map.addLayer(rivers.style(**{"color": "blue", "width": 1}), {}, "Rivers")

In [ ]:
chunk_size = 3000
count = rivers.size().getInfo()

for start in range(0, count, chunk_size):
    end = start + chunk_size
    print(f"Processing {start} → {end}")


    chunk = ee.FeatureCollection(
        rivers.toList(chunk_size, start)
    )

    stats = indices.reduceRegions(
        collection=chunk,
        reducer=ee.Reducer.mean(),
        scale=30
    )

    stats = stats.map(compute_risk).map(classify_risk)

    task = ee.batch.Export.table.toDrive(
        collection=stats,
        description=f"rivers_part_{start}",
        folder="CassiniProject",
        fileNamePrefix=f"rivers_part_{start}",
        fileFormat="GeoJSON"
    )

    task.start()

Processing 0 → 3000
Processing 3000 → 6000
Processing 6000 → 9000
Processing 9000 → 12000
Processing 12000 → 15000
Processing 15000 → 18000
Processing 18000 → 21000
Processing 21000 → 24000
Processing 24000 → 27000
Processing 27000 → 30000
Processing 30000 → 33000
Processing 33000 → 36000
